# Encoder–Decoder Transformer

A small English-to-French translation model built with TensorFlow and Keras.

## Load the data

In [ ]:
import numpy as np
import pandas as pd

dataset = pd.read_csv("english_french_1000_short_pairs.csv")

X = dataset["english"].values
Y = dataset["french"].values

print(dataset.shape)
dataset.head()


## Prepare the French text

In [ ]:
Y = np.array([f"start {sentence} end" for sentence in Y])


In [ ]:
from sklearn.model_selection import train_test_split

X_train_val,X_test,Y_train_val,Y_test=train_test_split(X,Y,test_size=0.10, random_state=42)

X_train,X_val,Y_train,Y_val=train_test_split(X_train_val,Y_train_val,test_size=1/9, random_state=42)


## Tokenize the sentences

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer_eng = Tokenizer(oov_token="unk")
tokenizer_eng.fit_on_texts(X_train)

tokenizer_fr = Tokenizer(oov_token="unk")
tokenizer_fr.fit_on_texts(Y_train)

X_train = tokenizer_eng.texts_to_sequences(X_train)
X_val = tokenizer_eng.texts_to_sequences(X_val)
X_test = tokenizer_eng.texts_to_sequences(X_test)

Y_train = tokenizer_fr.texts_to_sequences(Y_train)
Y_val = tokenizer_fr.texts_to_sequences(Y_val)
Y_test = tokenizer_fr.texts_to_sequences(Y_test)

eng_vocab_size = len(tokenizer_eng.word_index) + 1
fr_vocab_size = len(tokenizer_fr.word_index) + 1

print("English vocabulary:", eng_vocab_size)
print("French vocabulary:", fr_vocab_size)


## Pad the sequences

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_eng_len = max(
    max(len(sentence) for sentence in X_train),
    max(len(sentence) for sentence in X_val),
    max(len(sentence) for sentence in X_test)
)

max_fr_len = max(
    max(len(sentence) for sentence in Y_train),
    max(len(sentence) for sentence in Y_val),
    max(len(sentence) for sentence in Y_test)
)

X_train = pad_sequences(X_train, maxlen=max_eng_len, padding="post",truncating="post")
X_val   = pad_sequences(X_val,   maxlen=max_eng_len, padding="post",truncating="post")
X_test  = pad_sequences(X_test,  maxlen=max_eng_len, padding="post",truncating="post")

Y_train = pad_sequences(Y_train, maxlen=max_fr_len, padding="post",truncating="post")
Y_val   = pad_sequences(Y_val,   maxlen=max_fr_len, padding="post",truncating="post")
Y_test  = pad_sequences(Y_test,  maxlen=max_fr_len, padding="post",truncating="post")

decoder_input_train = Y_train[:, :-1]
decoder_target_train = Y_train[:, 1:]

decoder_input_val = Y_val[:, :-1]
decoder_target_val = Y_val[:, 1:]

decoder_input_test = Y_test[:, :-1]
decoder_target_test = Y_test[:, 1:]


In [ ]:
print("English:", X_train.shape)
print("Decoder input:", decoder_input_train.shape)
print("Decoder target:", decoder_target_train.shape)


## Model settings and masks

In [ ]:
import tensorflow as tf

d_model = 64

encoder_mask_train = tf.not_equal(X_train, 0)
decoder_mask_train = tf.not_equal(decoder_input_train, 0)

encoder_mask_val = tf.not_equal(X_val, 0)
decoder_mask_val = tf.not_equal(decoder_input_val, 0)

encoder_mask_test = tf.not_equal(X_test, 0)
decoder_mask_test = tf.not_equal(decoder_input_test, 0)


## Token and position embeddings

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Embedding

class PositionalEmbedding(Layer):
    def __init__(self, vocab_size, sequence_length, d_model):
        super().__init__()
        self.token_embedding = Embedding(
            input_dim=vocab_size,
            output_dim=d_model,
            mask_zero=False
        )
        self.position_embedding = Embedding(
            input_dim=sequence_length,
            output_dim=d_model
        )

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)

        token_vectors = self.token_embedding(inputs)
        position_vectors = self.position_embedding(positions)

        return token_vectors + position_vectors

eng_pos_embedding = PositionalEmbedding(
    eng_vocab_size, max_eng_len, d_model
)

fr_pos_embedding = PositionalEmbedding(
    fr_vocab_size, max_fr_len - 1, d_model
)


## Encoder block

In [ ]:
from tensorflow.keras.layers import (
    Dense, Dropout, LayerNormalization, MultiHeadAttention
)

ff_dim = 128
num_heads = 4
dropout_rate = 0.1

class EncoderBlock(Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads
        )

        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(d_model)
        ])

        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)

    def call(self, inputs, training=False, mask=None):
        attention_mask = None

        if mask is not None:
            attention_mask = mask[:, tf.newaxis, :]

        attention_output = self.attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=attention_mask,
            training=training
        )

        attention_output = self.dropout1(
            attention_output,
            training=training
        )

        x = self.norm1(inputs + attention_output)

        ffn_output = self.ffn(x)
        ffn_output = self.dropout2(ffn_output, training=training)

        return self.norm2(x + ffn_output)


encoder_block = EncoderBlock(
    d_model=d_model,
    num_heads=num_heads,
    ff_dim=ff_dim,
    dropout_rate=dropout_rate,
)


## Decoder block

In [ ]:
from tensorflow.keras.layers import (
    Layer, Dense, Dropout, LayerNormalization, MultiHeadAttention
)

class DecoderBlock(Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout_rate
        )

        self.cross_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout_rate
        )

        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(d_model)
        ])

        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.norm3 = LayerNormalization(epsilon=1e-6)

        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)
        self.dropout3 = Dropout(dropout_rate)

    def call(
        self,
        decoder_inputs,
        encoder_outputs,
        training=False,
        decoder_mask=None,
        encoder_mask=None
    ):
        decoder_attention_mask = None
        if decoder_mask is not None:
            decoder_attention_mask = decoder_mask[:, tf.newaxis, :]

        self_attention_output = self.self_attention(
            query=decoder_inputs,
            value=decoder_inputs,
            key=decoder_inputs,
            attention_mask=decoder_attention_mask,
            use_causal_mask=True,
            training=training
        )

        self_attention_output = self.dropout1(
            self_attention_output,
            training=training
        )

        x = self.norm1(decoder_inputs + self_attention_output)

        encoder_attention_mask = None
        if encoder_mask is not None:
            encoder_attention_mask = encoder_mask[:, tf.newaxis, :]

        cross_attention_output = self.cross_attention(
            query=x,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=encoder_attention_mask,
            training=training
        )

        cross_attention_output = self.dropout2(
            cross_attention_output,
            training=training
        )

        x = self.norm2(x + cross_attention_output)

        ffn_output = self.ffn(x)
        ffn_output = self.dropout3(ffn_output, training=training)

        return self.norm3(x + ffn_output)


decoder_block = DecoderBlock(
    d_model=d_model,
    num_heads=num_heads,
    ff_dim=ff_dim,
    dropout_rate=dropout_rate,
)


## Build the full model

In [ ]:
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model

english_input = Input(
    shape=(max_eng_len,),
    dtype="int32",
    name="english_input"
)

french_input = Input(
    shape=(max_fr_len - 1,),
    dtype="int32",
    name="french_decoder_input"
)

encoder_mask = Lambda(
    lambda x: tf.not_equal(x, 0),
    name="encoder_padding_mask"
)(english_input)

decoder_mask = Lambda(
    lambda x: tf.not_equal(x, 0),
    name="decoder_padding_mask"
)(french_input)

english_vectors = eng_pos_embedding(english_input)

encoder_output = encoder_block(
    english_vectors,
    mask=encoder_mask
)

french_vectors = fr_pos_embedding(french_input)

decoder_output = decoder_block(
    decoder_inputs=french_vectors,
    encoder_outputs=encoder_output,
    decoder_mask=decoder_mask,
    encoder_mask=encoder_mask
)

output_logits = Dense(
    fr_vocab_size,
    name="french_token_logits"
)(decoder_output)

model = Model(
    inputs=[english_input, french_input],
    outputs=output_logits,
    name="english_to_french_transformer"
)

model.summary()


## Loss and accuracy

In [ ]:
loss_function = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)

def masked_loss(y_true, y_pred):
    loss = loss_function(y_true, y_pred)

    mask = tf.cast(tf.not_equal(y_true, 0), loss.dtype)
    loss = loss * mask

    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

def masked_accuracy(y_true, y_pred):
    predicted_ids = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)

    matches = tf.cast(tf.equal(y_true, predicted_ids), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)

    matches = matches * mask

    return tf.reduce_sum(matches) / tf.reduce_sum(mask)


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=masked_loss,
    metrics=[masked_accuracy]
)


## Train the model

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    x={
        "english_input": X_train,
        "french_decoder_input": decoder_input_train
    },
    y=decoder_target_train,
    validation_data=(
        {
            "english_input": X_val,
            "french_decoder_input": decoder_input_val
        },
        decoder_target_val
    ),
    epochs=30,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=2
)


## Evaluate on the test set

In [ ]:
test_results = model.evaluate(
    x={
        "english_input": X_test,
        "french_decoder_input": decoder_input_test
    },
    y=decoder_target_test,
    return_dict=True
)

print(test_results)


## Translate new sentences

In [ ]:
def translate_english_to_french(english_sentence):
    english_sequence = tokenizer_eng.texts_to_sequences([english_sentence])

    english_sequence = pad_sequences(
        english_sequence,
        maxlen=max_eng_len,
        padding="post",
        truncating="post"
    )

    start_id = tokenizer_fr.word_index["start"]
    end_id = tokenizer_fr.word_index["end"]

    generated_ids = [start_id]

    for _ in range(max_fr_len - 1):
        french_sequence = pad_sequences(
            [generated_ids],
            maxlen=max_fr_len - 1,
            padding="post",
            truncating="post"
        )

        predictions = model.predict(
            {
                "english_input": english_sequence,
                "french_decoder_input": french_sequence
            },
            verbose=0
        )

        current_position = len(generated_ids) - 1
        next_id = int(
            tf.argmax(predictions[0, current_position, :]).numpy()
        )

        if next_id == end_id or next_id == 0:
            break

        generated_ids.append(next_id)

    french_words = [
        tokenizer_fr.index_word.get(token_id, "<unk>")
        for token_id in generated_ids[1:]
    ]

    return " ".join(french_words)


In [ ]:
print(translate_english_to_french("The cat is in the garden."))
print(translate_english_to_french("The student sees a phone."))


## Note

This uses a small template-based dataset, so it is meant to show how the architecture works rather than act as a general translation system.